In [1]:
import tempfile
import os

import tensorflow as tf
import tensorflow_model_optimization as tfmot
import numpy as np

import onnxruntime as ort
import tf2onnx
import onnx

from modules.datastore import load_datasets  # noqa: E402
from modules.helper_functions import display_samples,eval_model,show_incorrect_predictions  # noqa: E402

2024-07-16 01:26:17.178013: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9373] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-07-16 01:26:17.178433: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-07-16 01:26:17.245367: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1534] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-07-16 01:26:17.387718: I tensorflow/core/platform/cpu_feature_guard.cc:183] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX, in other operations, rebuild TensorFlow with the appropriate compiler flags.


ModuleNotFoundError: No module named 'tensorflow_model_optimization'

In [ ]:
# set up our initial variables
dataset_path = 'datasets/CompositeDS/'
class_map = {0:"Original", 1:"Poisoned"}
batch_size = 32
image_resolution = 512

# and load the dataset
ds = load_datasets(dataset_path, batch_size, image_resolution)

In [ ]:
# first prune the model
# see https://www.tensorflow.org/model_optimization/guide/pruning/pruning_with_keras
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

model = tf.keras.models.load_model('results/models/export/to_export.keras')

# Compute end step to finish pruning after 2 epochs.

epochs = 2
validation_split = 0.1 # 10% of training set will be used for validation set. 

num_images = ds['train'].shape[0] * (1 - validation_split)
end_step = np.ceil(num_images / batch_size).astype(np.int32) * epochs

# Define model for pruning.
pruning_params = {
      'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(initial_sparsity=0.50,
                                                               final_sparsity=0.80,
                                                               begin_step=0,
                                                               end_step=end_step)
}

model_for_pruning = prune_low_magnitude(model, **pruning_params)

# `prune_low_magnitude` requires a recompile.
model_for_pruning.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(),
              metrics=['accuracy'])

model_for_pruning.summary()


In [ ]:
# now run pruning
logdir = tempfile.mkdtemp()

callbacks = [
  tfmot.sparsity.keras.UpdatePruningStep(),
  tfmot.sparsity.keras.PruningSummaries(log_dir=logdir),
]

model_for_pruning.fit(ds['train'], batch_size=batch_size, epochs=epochs, callbacks=callbacks, validation_data=ds['validation'])

# and save the model
model_for_pruning.save('results/models/export/pruned.keras')

In [ ]:
# finally export the model to ONNX for deployment
input_signature = [tf.TensorSpec([None, 512, 512, 3], tf.float32, name='x')]

# Use from_function for tf functions
onnx_model, _ = tf2onnx.convert.from_keras(model_for_pruning, input_signature, opset=13)
onnx.save(onnx_model, "results/models/export/exported.onnx")

In [ ]:
# verify
model = tf.keras.models.load_model('results/models/export/pruned.keras')

# Change shapes and types to match model
input1 = np.zeros((1, 512, 512, 3), np.float32)

# Start from ORT 1.10, ORT requires explicitly setting the providers parameter if you want to use execution providers
# other than the default CPU provider (as opposed to the previous behavior of providers getting set/registered by default
# based on the build flags) when instantiating InferenceSession.
# Following code assumes NVIDIA GPU is available, you can specify other execution providers or don't include providers parameter
# to use default CPU provider.
sess = ort.InferenceSession("results/models/export/exported.onnx", providers=["CUDAExecutionProvider"])

# Set first argument of sess.run to None to use all model outputs in default order
# Input/output names are printed by the CLI and can be set with --rename-inputs and --rename-outputs
# If using the python API, names are determined from function arg names or TensorSpec names.
results_ort = sess.run(["output1", "output2"], {"input1": input1})
results_tf = model(input1)

for ort_res, tf_res in zip(results_ort, results_tf):
    np.testing.assert_allclose(ort_res, tf_res, rtol=1e-5, atol=1e-5)

print("Results match")